# Stage 2.3 - Controlled five-fold decoder comparison

This notebook trains and evaluates a 2x2 factorial of four decoder arms using the byte-identical fold-specific front end exported by Stage 2.2. The arms span {ReLU, PReLU} activation x {plain, residual} skip topology:

- **U (plain_unet_style):** plain Conv3d-GroupNorm-ReLU refinement blocks.
- **V (residual_vnet_style):** projected residual Conv3d-GroupNorm-PReLU refinement blocks.
- **ReLU + residual (residual_relu_style):** projected residual Conv3d-GroupNorm-ReLU blocks.
- **PReLU + plain (plain_prelu_style):** plain Conv3d-GroupNorm-PReLU refinement blocks.

These are deliberately matched adaptations, not faithful reproductions of the complete original 3D U-Net and V-Net. Channel schedule, skip concatenation, transposed upsampling, final trilinear interpolation, loss, augmentation, optimiser, seed, and stopping rule are shared, and all four arms share identical conv/norm initialisation so only the residual/activation parameters differ. Completing the square lets Stage 2.4 estimate the residual and activation main effects (and their interaction) instead of confounding them in a single diagonal contrast.

## Success criteria

1. Every fold passes the independent encoder QA gate before decoder training starts.
2. All arms use the same `shared_frontend.pth` hash, tensors, split, augmentation seeds, training seed, shared conv/norm initialisation, and optimiser policy within a fold.
3. The shared front end remains in evaluation/inference mode, has no gradients, and has the same state hash before and after each run.
4. Each arm outputs `[B,4,256,256,256]` in `[femur,tibia,patella,fibula]` order; the only 256-cubed interpolation is shared four-channel trilinear upsampling with `align_corners=False`, and each run records its logit resolution and effective spacing.
5. Five folds by four arms produce twenty best checkpoints and exactly 71 held-out OOF records per arm; the optional fold-0 seed sweep adds a three-seed variance band.
6. Empty predictions score Dice zero and receive the physical volume-diagonal ASSD penalty rather than being dropped as `NaN`.
7. Every GPU run records peak memory and retains at least 10% device-memory headroom.

## Academic basis

[3D U-Net (Çiçek et al., 2016)](https://lmb.informatik.uni-freiburg.de/Publications/2016/CABR16/) and [V-Net (Milletari et al., 2016)](https://arxiv.org/abs/1606.04797) motivate the two block families. [UNETR (Hatamizadeh et al., 2022)](https://openaccess.thecvf.com/content/WACV2022/html/Hatamizadeh_UNETR_Transformers_for_3D_Medical_Image_Segmentation_WACV_2022_paper.html) directly reports a controlled decoder-choice ablation under one encoder. [Group Normalization (Wu and He, 2018)](https://openaccess.thecvf.com/content_ECCV_2018/html/Yuxin_Wu_Group_Normalization_ECCV_2018_paper.html) supports batch-size-one training. [EffiDec3D (Rahman and Marculescu, CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Rahman_EffiDec3D_An_Optimized_Decoder_for_High-Performance_and_Efficient_3D_Medical_CVPR_2025_paper.html) motivates removing expensive high-resolution decoder layers; this notebook applies one shared final trilinear interpolation as the memory-bounded adaptation.

In [ ]:
import contextlib
import hashlib
import json
import os
import platform
import random
try:
    import resource
except ImportError:
    resource = None
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

STAGE2_SCHEMA = "foundation_stage2_v1"
SEED = 42
LEARNING_RATE = 1e-3
USE_AMP = True
USE_ACTIVATION_CHECKPOINTING = True
DECODER_LOGIT_RESOLUTION = 128  # decoder emits four-channel logits at this cube, then one shared trilinear upsample to 256^3
BONES = ["femur", "tibia", "patella", "fibula"]
FEATURE_CHANNELS = [64, 128, 256, 512]
# 2x2 factorial: {ReLU, PReLU} activation x {plain, residual} skip topology. The two original
# arms sit on one diagonal; the two new arms complete the square so 04 can decompose the
# residual and activation main effects (and their interaction) instead of one confounded contrast.
ARM_FACTORS = {
    "plain_unet_style":    {"activation": "relu",  "residual": False, "label": "U (ReLU, plain)"},
    "residual_vnet_style": {"activation": "prelu", "residual": True,  "label": "V (PReLU, residual)"},
    "residual_relu_style": {"activation": "relu",  "residual": True,  "label": "ReLU + residual"},
    "plain_prelu_style":   {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
}
ARMS = list(ARM_FACTORS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(DEVICE)})

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt, label


def overlap_metrics(prediction, target):
    """Explicit empty handling: an empty target is invalid; an empty prediction against a target scores zero."""
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    invalid_target = not target.any(); empty_prediction = not prediction.any()
    if invalid_target:
        return {"dice": float("nan"), "iou": float("nan"), "invalid_target": True, "empty_prediction": empty_prediction}
    if empty_prediction:
        return {"dice": 0.0, "iou": 0.0, "invalid_target": False, "empty_prediction": True}
    intersection = np.logical_and(prediction, target).sum(dtype=np.float64)
    pred_count = prediction.sum(dtype=np.float64); target_count = target.sum(dtype=np.float64)
    return {"dice": float(2 * intersection / (pred_count + target_count)), "iou": float(intersection / (pred_count + target_count - intersection)), "invalid_target": False, "empty_prediction": False}


def _surface_distances(prediction, target, spacing_xyz):
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    spacing_xyz = tuple(float(value) for value in spacing_xyz)
    if len(spacing_xyz) != 3 or any(value <= 0 for value in spacing_xyz): raise ValueError("spacing_xyz must contain three positive millimetre values")
    pred_surface = prediction & ~binary_erosion(prediction); target_surface = target & ~binary_erosion(target)
    if not pred_surface.any() or not target_surface.any(): return None
    to_target = distance_transform_edt(~target_surface, sampling=spacing_xyz)[pred_surface]
    to_prediction = distance_transform_edt(~pred_surface, sampling=spacing_xyz)[target_surface]
    return np.concatenate([to_target, to_prediction])


def assd_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(distances.mean())


def two_largest_components(mask):
    labels, count = label(np.asarray(mask, dtype=bool))
    if count < 2: return None
    sizes = [(component, int((labels == component).sum())) for component in range(1, count + 1)]
    selected = sorted(sizes, key=lambda item: item[1], reverse=True)[:2]
    return labels == selected[0][0], labels == selected[1][0]


def minimum_component_gap_mm(mask, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    pair = two_largest_components(mask)
    if pair is None: return float("nan")
    first, second = pair
    return float(distance_transform_edt(~second, sampling=spacing_xyz)[first].min())


def component_bridge_metrics(prediction, target, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    _, pred_components = label(np.asarray(prediction, dtype=bool)); _, target_components = label(np.asarray(target, dtype=bool))
    return {"prediction_components": int(pred_components), "target_components": int(target_components), "component_agreement": bool(pred_components == target_components), "false_bridge": bool(target_components >= 2 and pred_components < target_components), "prediction_min_gap_mm": minimum_component_gap_mm(prediction, spacing_xyz), "target_min_gap_mm": minimum_component_gap_mm(target, spacing_xyz)}

In [ ]:
def find_project_root(start):
    for candidate in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (candidate / "configs" / "baseline_protocol_v1.json").exists(): return candidate
    raise FileNotFoundError("project root not found")


ROOT = find_project_root(Path.cwd())
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""): h.update(chunk)
    return h.hexdigest()


def tensor_sha256(tensor):
    value = tensor.detach().cpu().contiguous(); h = hashlib.sha256(); h.update(str(value.dtype).encode()); h.update(np.asarray(value.shape, dtype=np.int64).tobytes()); h.update(value.numpy().tobytes()); return h.hexdigest()


def canonical_sha256(payload): return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def state_sha256(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous(); digest.update(name.encode()); digest.update(str(tensor.dtype).encode()); digest.update(np.asarray(tensor.shape, dtype=np.int64).tobytes()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def peak_host_memory_bytes():
    if resource is None: return None
    maximum_rss = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    return maximum_rss if platform.system() == "Darwin" else maximum_rss * 1024


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_activation(kind, channels):
    if kind == "relu":
        return nn.ReLU(inplace=True)
    if kind == "prelu":
        return nn.PReLU(channels)
    raise ValueError(f"unknown activation {kind!r}")  # never silently fall through to PReLU


class DecoderBlock(nn.Module):
    """Unified refinement block. Residual skip and activation are the ONLY axes that vary across arms,
    so the four arms form a clean 2x2. With residual=False, activation=relu it reproduces the original
    PlainDoubleConv; with residual=True, activation=prelu it reproduces the original ResidualVNetBlock
    (verified numerically in block_equivalence_test)."""
    def __init__(self, input_channels, output_channels, activation, residual):
        super().__init__()
        self.residual = residual
        self.proj = (nn.Conv3d(input_channels, output_channels, 1) if input_channels != output_channels else nn.Identity()) if residual else None
        self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1); self.norm1 = nn.GroupNorm(8, output_channels); self.act1 = make_activation(activation, output_channels)
        self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1); self.norm2 = nn.GroupNorm(8, output_channels); self.act2 = make_activation(activation, output_channels)
    def forward(self, x):
        residual = self.proj(x) if self.residual else None
        x = self.act1(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        if self.residual:
            x = x + residual
        return self.act2(x)


def block_for(arm, input_channels, output_channels):
    if arm not in ARM_FACTORS:
        raise ValueError(f"unknown arm {arm}")
    factors = ARM_FACTORS[arm]
    return DecoderBlock(input_channels, output_channels, factors["activation"], factors["residual"])


def dice_bce_loss(logits, target):
    logits = logits.float(); target = target.float(); bce = F.binary_cross_entropy_with_logits(logits, target); probability = torch.sigmoid(logits).flatten(2); flattened = target.flatten(2); intersection = (probability * flattened).sum(-1); dice = (2 * intersection + 1.0) / (probability.sum(-1) + flattened.sum(-1) + 1.0); return 0.5 * bce + 0.5 * (1 - dice.mean())


@torch.no_grad()
def hard_dice(logits, target):
    prediction = (torch.sigmoid(logits.float()) > 0.5).float().flatten(2); target = (target > 0.5).float().flatten(2); intersection = (prediction * target).sum(-1); return (2 * intersection + 1e-6) / (prediction.sum(-1) + target.sum(-1) + 1e-6)

In [ ]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)


def make_scaler(): return torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported())


In [ ]:
# Full controlled cross-validation implementation.
import nibabel as nib
import timm
from torch.utils.data import DataLoader, Dataset, get_worker_info

RUN_CROSS_VALIDATION = False
RUN_SEED_SWEEP = False
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
SWEEP_FOLD = 0
SWEEP_SEEDS = [123, 2024]  # seed 42 is reused from the main-CV fold-0 runs to complete the 3-seed band
MAX_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 1
NUM_WORKERS = 4 if os.name != "nt" else 0
SAVE_OOF_MASKS = True
SPACING_XYZ = (0.78125, 0.78125, 0.78125)
ARM_LABELS = {arm: ARM_FACTORS[arm]["label"] for arm in ARM_FACTORS}
PRETRAIN_MODEL = "convnextv2_tiny.fcmae"
FUSION_TYPES = ["local", "local", "attention", "attention"]
AUGMENTATION = {"gamma": [0.90, 1.10], "brightness": [-0.05, 0.05], "gaussian_noise_sigma": [0.0, 0.02]}
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MANIFEST_META_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.metadata.json"
DATA_CONFIG_PATH = ROOT / "configs" / "data_contract_v1.json"


def stable_seed(*parts):
    token = "|".join(str(part) for part in parts).encode()
    return int.from_bytes(hashlib.sha256(token).digest()[:8], "little") % (2**32)


def deterministic_augment(array, sample_id, view, epoch, worker_id, seed):
    rng = np.random.default_rng(stable_seed(seed, sample_id, view, epoch, worker_id))
    gamma = rng.uniform(*AUGMENTATION["gamma"]); brightness = rng.uniform(*AUGMENTATION["brightness"]); sigma = rng.uniform(*AUGMENTATION["gaussian_noise_sigma"])
    output = np.power(np.clip(array, 0, 1), gamma, dtype=np.float32) + np.float32(brightness)
    if sigma > 0:
        output += rng.normal(0, sigma, output.shape).astype(np.float32)
    return np.clip(output, 0, 1).astype(np.float32)


def load_fold_rows(fold):
    meta = json.loads(MANIFEST_META_PATH.read_text(encoding="utf-8"))
    if not meta.get("certification_approved", False) or sha256_file(MANIFEST_PATH) != meta.get("sha256"):
        raise RuntimeError("certified manifest gate failed")
    rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    rows = rows[rows.status.eq("ready")].copy(); rows["test_fold"] = rows.test_fold.astype(int)
    data_contract = json.loads(DATA_CONFIG_PATH.read_text(encoding="utf-8"))
    if rows.target_version.ne(data_contract["target_version"]).any() or rows.drr_version.ne(data_contract["drr_version"]).any():
        raise RuntimeError("certified manifest data-version mismatch")
    if len(rows) != 71 or rows.groupby("subject_id").test_fold.nunique().max() != 1:
        raise RuntimeError("71-knee subject-level fold contract failed")
    rows["split"] = "train"
    rows.loc[rows.test_fold.eq(fold), "split"] = "test"
    rows.loc[rows.test_fold.eq((fold + 1) % 5), "split"] = "validation"
    subjects = {name: set(group.subject_id) for name, group in rows.groupby("split")}
    if subjects["train"] & subjects["validation"] or subjects["train"] & subjects["test"] or subjects["validation"] & subjects["test"]:
        raise RuntimeError("subject leakage between fold roles")
    return {name: rows[rows.split.eq(name)].copy() for name in ("train", "validation", "test")}, meta


def read_drr(path):
    array = np.load(path).astype(np.float32)
    if array.shape != (256, 256) or not np.isfinite(array).all():
        raise ValueError(f"invalid DRR: {path}")
    return np.clip(array, 0, 1)


def load_target(row):
    arrays = []
    for bone in BONES:
        image = nib.load(str(ROOT / row.target_path / f"{row.sample_id}_{bone}.nii.gz"))
        if image.shape != (256, 256, 256) or tuple(nib.aff2axcodes(image.affine)) != ("L", "P", "S"):
            raise ValueError(f"target geometry mismatch: {row.sample_id}/{bone}")
        array = np.asarray(image.dataobj, dtype=np.float32)
        if array.sum() == 0 or not set(np.unique(array).tolist()).issubset({0.0, 1.0}):
            raise ValueError(f"invalid target: {row.sample_id}/{bone}")
        arrays.append(array)
    return torch.from_numpy(np.stack(arrays).astype(np.float32))


class ReconstructionDataset(Dataset):
    def __init__(self, rows, training, seed):
        self.rows = rows.reset_index(drop=True); self.training = training; self.seed = seed; self.epoch = 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]; worker = get_worker_info(); worker_id = 0 if worker is None else worker.id
        ap = read_drr(ROOT / row.ap_drr_path); lat = read_drr(ROOT / row.lat_drr_path)
        if self.training:
            ap = deterministic_augment(ap, row.sample_id, "ap", self.epoch, worker_id, self.seed); lat = deterministic_augment(lat, row.sample_id, "lat", self.epoch, worker_id, self.seed)
        return {"ap": torch.from_numpy(ap).unsqueeze(0), "lat": torch.from_numpy(lat).unsqueeze(0), "target": load_target(row), "sample_id": row.sample_id, "subject_id": row.subject_id, "dataset": row.dataset, "test_fold": int(row.test_fold)}


class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.query = nn.Linear(dim, dim); self.key = nn.Linear(dim, dim); self.value = nn.Linear(dim, dim); self.scale = dim ** -0.5
    def forward(self, query_map, context_map):
        batch, channels, height, width = query_map.shape; query = query_map.flatten(2).transpose(1, 2); context = context_map.flatten(2).transpose(1, 2)
        attention = torch.softmax(self.query(query) @ self.key(context).transpose(-2, -1) * self.scale, dim=-1)
        return (attention @ self.value(context) + query).transpose(1, 2).reshape(batch, channels, height, width)


class LocalFusion(nn.Module):
    def __init__(self, dim): super().__init__(); self.mix = nn.Conv2d(2 * dim, dim, 3, padding=1)
    def forward(self, query_map, context_map): return self.mix(torch.cat([query_map, context_map], dim=1)) + query_map


class BiPlanarFrontEnd(nn.Module):
    def __init__(self, encoder_state, pretrained_configuration):
        super().__init__(); self.encoder = timm.create_model(PRETRAIN_MODEL, pretrained=False, features_only=True); self.encoder.load_state_dict(encoder_state, strict=True); self.pretrained_configuration = pretrained_configuration
        channels = self.encoder.feature_info.channels(); self.fusion = nn.ModuleList([CrossAttention(c) if kind == "attention" else LocalFusion(c) for c, kind in zip(channels, FUSION_TYPES)]); self.project_2d = nn.ModuleList([nn.Conv2d(source, target, 1) for source, target in zip(channels, FEATURE_CHANNELS)]); self.fuse_3d = nn.ModuleList([nn.Conv3d(2 * c, c, 3, padding=1) for c in FEATURE_CHANNELS])
    def normalize(self, raw):
        image = raw.repeat(1, 3, 1, 1); mean = torch.as_tensor(self.pretrained_configuration["mean"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1); std = torch.as_tensor(self.pretrained_configuration["std"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1); return (image - mean) / std
    @staticmethod
    def lift(ap_feature, lat_feature, projection, fusion3d):
        ap = projection(ap_feature); lat = projection(lat_feature).flip(3); batch, channels, size, _ = ap.shape
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(batch, channels, size, size, size); lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(batch, channels, size, size, size)
        return fusion3d(torch.cat([ap_cube, lat_cube], dim=1))
    def forward(self, ap_raw, lat_raw):
        ap_levels = self.encoder(self.normalize(ap_raw)); lat_levels = self.encoder(self.normalize(lat_raw)); output = []
        for ap, lat, fusion, project, fuse3d in zip(ap_levels, lat_levels, self.fusion, self.project_2d, self.fuse_3d):
            output.append(self.lift(fusion(ap, lat), fusion(lat, ap), project, fuse3d))
        return output


def require_independent_qa(fold_root, fold):
    audit_path = fold_root / "feature_audit" / "p2" / "feature_audit_summary.json"; audit = json.loads(audit_path.read_text(encoding="utf-8"))
    if audit.get("structural_status") != "PASS" or not audit.get("encoder_unchanged") or audit.get("test_paths_opened") is not False:
        raise RuntimeError(f"fold {fold} structural P2 audit failed")
    independent_path = fold_root / "feature_audit" / "independent_qa_verdict.json"
    independent = json.loads(independent_path.read_text(encoding="utf-8")) if independent_path.is_file() else None
    passed = audit.get("quality_status") == "PASS" or (independent and independent.get("fold") == fold and independent.get("status") == "PASS")
    if not passed:
        raise RuntimeError(f"fold {fold} independent Stage 2 QA PASS is required; current quality={audit.get('quality_status')}")
    return {"audit_sha256": sha256_file(audit_path), "quality_status": audit.get("quality_status"), "independent_verdict": independent, "independent_sha256": sha256_file(independent_path) if independent_path.is_file() else None}


def load_frozen_frontend(fold):
    fold_root = ROOT / "models" / STAGE2_SCHEMA / f"fold_{fold}"; qa = require_independent_qa(fold_root, fold)
    p2_path = fold_root / "fcmae_p2_encoder.pth"; config_path = fold_root / "fcmae_p1_config.json"; shared_path = fold_root / "shared_frontend.pth"; provenance_path = fold_root / "shared_frontend_provenance.json"
    for required in (p2_path, config_path, shared_path, provenance_path):
        if not required.is_file(): raise FileNotFoundError(required)
    p2 = torch.load(p2_path, map_location="cpu", weights_only=False); configuration = json.loads(config_path.read_text(encoding="utf-8"))["pretrained_configuration"]; shared = torch.load(shared_path, map_location="cpu", weights_only=False); provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
    if shared.get("fold") != fold or shared.get("stage") != "shared_frontend" or shared.get("p2_encoder_sha256") != sha256_file(p2_path) or provenance.get("checkpoint_sha256") != sha256_file(shared_path):
        raise RuntimeError(f"fold {fold} shared-front-end provenance mismatch")
    front_end = BiPlanarFrontEnd(p2["encoder_state"], configuration); front_end.load_state_dict(shared["front_end_state"], strict=True)
    for parameter in front_end.parameters(): parameter.requires_grad = False
    return front_end.to(DEVICE).eval(), shared, provenance, qa


class MatchedDecoder3D(nn.Module):
    """Matched decoder with a shared low-memory output policy: logits at DECODER_LOGIT_RESOLUTION,
    then one shared trilinear upsample to 256^3. The 256^3 ASSD is therefore bounded by the logit
    resolution (recorded as effective_spacing_mm), not by the reported 0.78125 mm target spacing."""
    def __init__(self, arm):
        super().__init__(); self.arm = arm; c0, c1, c2, c3 = FEATURE_CHANNELS
        self.up3 = nn.ConvTranspose3d(c3, c2, 2, 2); self.dec3 = block_for(arm, c2 + c2, c2)
        self.up2 = nn.ConvTranspose3d(c2, c1, 2, 2); self.dec2 = block_for(arm, c1 + c1, c1)
        self.up1 = nn.ConvTranspose3d(c1, c0, 2, 2); self.dec1 = block_for(arm, c0 + c0, c0)
        self.refine128 = block_for(arm, c0, 32); self.output128 = nn.Conv3d(32, len(BONES), 1)
    def _run(self, module, value):
        if USE_ACTIVATION_CHECKPOINTING and self.training and value.requires_grad: return checkpoint.checkpoint(module, value, use_reentrant=False)
        return module(value)
    def forward(self, features):
        l0, l1, l2, l3 = features
        value = self._run(self.dec3, torch.cat([self.up3(l3), l2], 1)); value = self._run(self.dec2, torch.cat([self.up2(value), l1], 1)); value = self._run(self.dec1, torch.cat([self.up1(value), l0], 1))
        resolution = DECODER_LOGIT_RESOLUTION
        value = self._run(self.refine128, F.interpolate(value, size=(resolution, resolution, resolution), mode="trilinear", align_corners=False)); logits = self.output128(value)
        return F.interpolate(logits, size=(256, 256, 256), mode="trilinear", align_corners=False)


Decoder3D = MatchedDecoder3D

SHARED_INIT_EXCLUDE = (".proj.", ".act1.", ".act2.")  # arm-specific parameters (residual projection, PReLU)


def shared_init_state(model):
    return {name: tensor for name, tensor in model.state_dict().items() if not any(token in name for token in SHARED_INIT_EXCLUDE)}


def build_matched_decoder(arm, seed):
    """Give every arm identical conv/norm/up/output initialisation for a given seed so random init is
    not a hidden confound in the 2x2; only the intended proj/activation parameters differ. Returns the
    decoder plus a hash of its shared-layer init (equal across arms for the same seed)."""
    seed_everything(seed)
    reference = Decoder3D(ARMS[0])
    decoder = Decoder3D(arm)
    reference_shared = shared_init_state(reference)
    own = decoder.state_dict()
    transferable = {name: reference_shared[name] for name in own if name in reference_shared and own[name].shape == reference_shared[name].shape}
    decoder.load_state_dict(transferable, strict=False)
    return decoder, state_sha256(shared_init_state(decoder))


def architecture_contract():
    result = {}
    for arm in ARMS:
        model = Decoder3D(arm)
        group_norms = [module for module in model.modules() if isinstance(module, nn.GroupNorm)]
        if len(group_norms) != 8:
            raise RuntimeError(f"GroupNorm count contract failed for {arm}: {len(group_norms)}")
        for group_norm in group_norms:
            if group_norm.num_groups != 8 or group_norm.num_channels % 8 != 0:
                raise RuntimeError(f"GroupNorm config contract failed for {arm}: groups={group_norm.num_groups} channels={group_norm.num_channels}")
        if any(isinstance(module, (nn.BatchNorm3d, nn.InstanceNorm3d)) for module in model.modules()):
            raise RuntimeError(f"forbidden batch/instance normalization present in {arm}")
        factors = ARM_FACTORS[arm]
        result[arm] = {"label": factors["label"], "activation": factors["activation"], "residual": factors["residual"], "parameters": sum(parameter.numel() for parameter in model.parameters()), "groupnorm_layers": len(group_norms), "output_policy": f"four_logits_{DECODER_LOGIT_RESOLUTION}_then_trilinear_256_align_corners_false"}
    return result


def metrics_with_empty_penalty(logits, target, spacing_xyz=SPACING_XYZ):
    prediction = torch.sigmoid(logits.float()).detach().cpu().numpy() > 0.5; truth = target.detach().cpu().numpy() > 0.5; rows = []
    for batch_index in range(prediction.shape[0]):
        row = {}
        for bone_index, bone in enumerate(BONES):
            pred = prediction[batch_index, bone_index]; actual = truth[batch_index, bone_index]; overlap = overlap_metrics(pred, actual)
            penalty = float(np.linalg.norm(np.asarray(actual.shape, dtype=float) * np.asarray(spacing_xyz, dtype=float)))
            distances = _surface_distances(pred, actual, spacing_xyz)
            row[f"dice_{bone}"] = overlap["dice"]; row[f"iou_{bone}"] = overlap["iou"]; row[f"assd_mm_{bone}"] = penalty if distances is None else float(distances.mean()); row[f"hd95_mm_{bone}"] = penalty if distances is None else float(np.percentile(distances, 95)); row[f"empty_prediction_{bone}"] = overlap["empty_prediction"]
            bridge = component_bridge_metrics(pred, actual, spacing_xyz); row[f"prediction_components_{bone}"] = bridge["prediction_components"]; row[f"target_components_{bone}"] = bridge["target_components"]; row[f"false_bridge_{bone}"] = bridge["false_bridge"]
        row["dice_macro"] = float(np.mean([row[f"dice_{bone}"] for bone in BONES])); row["assd_mm_macro"] = float(np.mean([row[f"assd_mm_{bone}"] for bone in BONES])); rows.append(row)
    return rows


def evaluate_validation(front_end, decoder, loader):
    decoder.eval(); values = []
    with torch.no_grad():
        for batch in loader:
            ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); target = batch["target"].to(DEVICE)
            with torch.no_grad(): features = front_end(ap, lat)
            with amp_context(): logits = decoder([feature.detach() for feature in features])
            values.append(hard_dice(logits, target).cpu())
    return float(torch.cat(values, dim=0).mean().item())


def run_fold_arm(fold, arm, seed=SEED, sweep=False):
    decoder, shared_init_sha = build_matched_decoder(arm, seed); decoder = decoder.to(DEVICE)
    splits, manifest_meta = load_fold_rows(fold); front_end, shared, provenance, qa = load_frozen_frontend(fold); front_hash_before = state_sha256(front_end.state_dict())
    if sweep:
        output_dir = ROOT / "models" / "decoders" / STAGE2_SCHEMA / "seed_sweep" / f"fold_{fold}" / arm / f"seed_{seed}"
    else:
        output_dir = ROOT / "models" / "decoders" / STAGE2_SCHEMA / f"fold_{fold}" / arm
    output_dir.mkdir(parents=True, exist_ok=True); mask_dir = output_dir / "oof_masks"; mask_dir.mkdir(exist_ok=True)
    generator = torch.Generator().manual_seed(seed); train_set = ReconstructionDataset(splits["train"], True, seed); validation_set = ReconstructionDataset(splits["validation"], False, seed); test_set = ReconstructionDataset(splits["test"], False, seed)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=NUM_WORKERS); validation_loader = DataLoader(validation_set, batch_size=1, shuffle=False, num_workers=NUM_WORKERS); test_loader = DataLoader(test_set, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
    optimizer = torch.optim.Adam(decoder.parameters(), lr=LEARNING_RATE); scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS); scaler = make_scaler(); history = []; best_dice, stale = -1.0, 0
    effective_spacing_mm = round(0.78125 * 256 / DECODER_LOGIT_RESOLUTION, 6)
    config = {"schema_version": STAGE2_SCHEMA, "stage": "controlled_decoder_cv", "fold": fold, "arm": arm, "arm_label": ARM_LABELS[arm], "activation": ARM_FACTORS[arm]["activation"], "residual": ARM_FACTORS[arm]["residual"], "seed": seed, "sweep": sweep, "logit_resolution": DECODER_LOGIT_RESOLUTION, "output_resolution": 256, "effective_spacing_mm": effective_spacing_mm, "shared_init_sha256": shared_init_sha, "manifest_sha256": manifest_meta["sha256"], "shared_frontend_sha256": sha256_file(ROOT / "models" / STAGE2_SCHEMA / f"fold_{fold}" / "shared_frontend.pth"), "shared_frontend_state_sha256": front_hash_before, "qa": qa, "architecture": architecture_contract()[arm], "hyperparameters": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE, "optimizer": "Adam", "learning_rate": LEARNING_RATE, "scheduler": "CosineAnnealingLR", "loss": "0.5_bce+0.5_soft_dice", "amp": USE_AMP, "activation_checkpointing": USE_ACTIVATION_CHECKPOINTING, "threshold": 0.5}}
    config_sha = canonical_sha256(config); (output_dir / "config.json").write_text(json.dumps(config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    started = time.time()
    for epoch in range(MAX_EPOCHS):
        train_set.set_epoch(epoch); decoder.train(); losses = []
        for batch in train_loader:
            ap = batch["ap"].to(DEVICE, non_blocking=True); lat = batch["lat"].to(DEVICE, non_blocking=True); target = batch["target"].to(DEVICE, non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.no_grad(): features = front_end(ap, lat)
            with amp_context(): logits = decoder([feature.detach() for feature in features]); loss = dice_bce_loss(logits, target)
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite loss fold={fold} arm={arm}")
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0); scaler.step(optimizer); scaler.update(); losses.append(loss.item())
        scheduler.step(); validation_dice = evaluate_validation(front_end, decoder, validation_loader); row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "validation_macro_dice": validation_dice, "lr": optimizer.param_groups[0]["lr"]}; history.append(row); pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False); print(row)
        if validation_dice > best_dice:
            best_dice, stale = validation_dice, 0; torch.save({"schema_version": STAGE2_SCHEMA, "stage": "controlled_decoder_cv", "fold": fold, "arm": arm, "seed": seed, "config_sha256": config_sha, "shared_frontend_sha256": config["shared_frontend_sha256"], "decoder_state": {key: value.detach().cpu() for key, value in decoder.state_dict().items()}}, output_dir / "best_decoder.pth")
        else: stale += 1
        if stale >= PATIENCE: break
    best = torch.load(output_dir / "best_decoder.pth", map_location="cpu", weights_only=False); decoder.load_state_dict(best["decoder_state"], strict=True); decoder.to(DEVICE).eval(); records = []
    with torch.no_grad():
        for batch in test_loader:
            ap = batch["ap"].to(DEVICE); lat = batch["lat"].to(DEVICE); target = batch["target"].to(DEVICE)
            with torch.no_grad(): features = front_end(ap, lat)
            with amp_context(): logits = decoder([feature.detach() for feature in features])
            metric = metrics_with_empty_penalty(logits, target)[0]; sample_id = batch["sample_id"][0]; dataset = batch["dataset"][0]
            record = {"sample_id": sample_id, "subject_id": batch["subject_id"][0], "dataset": dataset, "cohort": "fractured" if dataset == "Ruikar" else "healthy", "test_fold": fold, "arm": arm, "arm_label": ARM_LABELS[arm], "seed": seed, "logit_resolution": DECODER_LOGIT_RESOLUTION, "effective_spacing_mm": effective_spacing_mm, "shared_frontend_sha256": config["shared_frontend_sha256"], **metric}; records.append(record)
            if SAVE_OOF_MASKS: np.savez_compressed(mask_dir / f"{sample_id}.npz", prediction=(torch.sigmoid(logits.float())[0] > 0.5).cpu().numpy().astype(np.uint8), bones=np.asarray(BONES))
    frame = pd.DataFrame(records); frame.to_csv(output_dir / "oof_metrics.csv", index=False)
    front_hash_after = state_sha256(front_end.state_dict())
    if front_hash_before != front_hash_after or any(parameter.grad is not None for parameter in front_end.parameters()): raise RuntimeError("shared front end changed during decoder training")
    peak = int(torch.cuda.max_memory_allocated()) if DEVICE.type == "cuda" else None; total = int(torch.cuda.get_device_properties(DEVICE).total_memory) if DEVICE.type == "cuda" else None; headroom = None if total is None else 1.0 - peak / total
    if headroom is not None and headroom < 0.10: raise RuntimeError(f"GPU headroom below 10%: {headroom:.3f}")
    summary = {**config, "config_sha256": config_sha, "best_validation_macro_dice": best_dice, "epochs_completed": len(history), "oof_rows": len(frame), "front_end_state_sha256_after": front_hash_after, "front_end_unchanged": front_hash_before == front_hash_after, "checkpoint_sha256": sha256_file(output_dir / "best_decoder.pth"), "resource_usage": {"wall_seconds": round(time.time() - started, 1), "peak_gpu_bytes": peak, "total_gpu_bytes": total, "gpu_headroom_fraction": headroom, "peak_host_bytes": peak_host_memory_bytes()}, "success": len(frame) == len(splits["test"]) and front_hash_before == front_hash_after and (headroom is None or headroom >= 0.10)}
    (output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"); return summary


def metric_unit_tests():
    target = np.zeros((8, 8, 8), dtype=bool); target[2:6, 2:6, 2:6] = True
    if overlap_metrics(target, target)["dice"] != 1.0 or assd_mm(target, target, SPACING_XYZ) != 0.0: raise AssertionError("identical-mask metric test failed")
    empty = overlap_metrics(np.zeros_like(target), target)
    if empty["dice"] != 0.0: raise AssertionError("empty-prediction Dice test failed")
    expected_penalty = float(np.linalg.norm(np.asarray(target.shape) * np.asarray(SPACING_XYZ)))
    logits = torch.full((1, len(BONES), *target.shape), -100.0); targets = torch.from_numpy(np.stack([target] * len(BONES))[None]).float()
    penalized = metrics_with_empty_penalty(logits, targets, SPACING_XYZ)[0]
    if penalized["dice_macro"] != 0.0 or not np.isclose(penalized["assd_mm_macro"], expected_penalty): raise AssertionError("physical diagonal penalty test failed")
    return {"identical_dice": 1.0, "identical_assd_mm": 0.0, "empty_dice": penalized["dice_macro"], "empty_assd_penalty_mm": penalized["assd_mm_macro"]}


def block_equivalence_test():
    """Prove the unified DecoderBlock is numerically identical to the original hand-written blocks
    (not merely equal in parameter count). Legacy definitions are local and discarded on return."""
    class LegacyPlainDoubleConv(nn.Module):
        def __init__(self, input_channels, output_channels):
            super().__init__(); self.block = nn.Sequential(nn.Conv3d(input_channels, output_channels, 3, padding=1), nn.GroupNorm(8, output_channels), nn.ReLU(inplace=True), nn.Conv3d(output_channels, output_channels, 3, padding=1), nn.GroupNorm(8, output_channels), nn.ReLU(inplace=True))
        def forward(self, x): return self.block(x)

    class LegacyResidualVNetBlock(nn.Module):
        def __init__(self, input_channels, output_channels):
            super().__init__(); self.projection = nn.Conv3d(input_channels, output_channels, 1) if input_channels != output_channels else nn.Identity(); self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1); self.norm1 = nn.GroupNorm(8, output_channels); self.act1 = nn.PReLU(output_channels); self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1); self.norm2 = nn.GroupNorm(8, output_channels); self.act2 = nn.PReLU(output_channels)
        def forward(self, x):
            residual = self.projection(x); x = self.act1(self.norm1(self.conv1(x))); x = self.norm2(self.conv2(x)); return self.act2(x + residual)

    torch.manual_seed(0)
    x = torch.randn(1, 64, 8, 8, 8)
    legacy_plain = LegacyPlainDoubleConv(64, 64).eval(); new_plain = DecoderBlock(64, 64, "relu", False).eval()
    remap_plain = {"conv1.weight": legacy_plain.block[0].weight, "conv1.bias": legacy_plain.block[0].bias, "norm1.weight": legacy_plain.block[1].weight, "norm1.bias": legacy_plain.block[1].bias, "conv2.weight": legacy_plain.block[3].weight, "conv2.bias": legacy_plain.block[3].bias, "norm2.weight": legacy_plain.block[4].weight, "norm2.bias": legacy_plain.block[4].bias}
    new_plain.load_state_dict(remap_plain, strict=True)
    if not torch.allclose(legacy_plain(x), new_plain(x), atol=1e-6): raise AssertionError("plain-arm equivalence failed")
    xv = torch.randn(1, 128, 8, 8, 8)
    legacy_v = LegacyResidualVNetBlock(128, 64).eval(); new_v = DecoderBlock(128, 64, "prelu", True).eval()
    remap_v = {("proj." + name.split("projection.", 1)[1]) if name.startswith("projection.") else name: value for name, value in legacy_v.state_dict().items()}
    new_v.load_state_dict(remap_v, strict=True)
    if not torch.allclose(legacy_v(xv), new_v(xv), atol=1e-6): raise AssertionError("residual-arm equivalence failed")
    return {"plain_equivalent": True, "residual_equivalent": True}


def matched_init_check(seed=SEED):
    decoders = {}; hashes = {}
    for arm in ARMS:
        decoder, shared_sha = build_matched_decoder(arm, seed); decoders[arm] = decoder; hashes[arm] = shared_sha
    reference = shared_init_state(decoders[ARMS[0]])
    for arm in ARMS[1:]:
        state = shared_init_state(decoders[arm])
        if state.keys() != reference.keys(): raise AssertionError(f"shared-init key mismatch for {arm}")
        for key in reference:
            if not torch.equal(state[key], reference[key]): raise AssertionError(f"shared init differs at {key} for {arm}")
    if len(set(hashes.values())) != 1: raise AssertionError(f"shared-init hashes differ across arms: {hashes}")
    return {"arms": len(ARMS), "shared_init_sha256": next(iter(hashes.values()))}


def guard_tests():
    for bad in ("gelu", "", "RELU"):
        try:
            make_activation(bad, 8); raise AssertionError(f"make_activation accepted {bad!r}")
        except ValueError:
            pass
    try:
        block_for("nonexistent_arm", 8, 8); raise AssertionError("block_for accepted an unknown arm")
    except ValueError:
        pass
    return {"make_activation_guarded": True, "block_for_guarded": True}


def run_cross_validation():
    summaries = []
    for fold in FOLDS_TO_RUN:
        for arm in ARMS: summaries.append(run_fold_arm(fold, arm))
    frame = pd.DataFrame(summaries); expected = {(fold, arm) for fold in FOLDS_TO_RUN for arm in ARMS}; observed = set(zip(frame.fold, frame.arm))
    if observed != expected or not frame.success.all(): raise RuntimeError("twenty-run cross-validation contract failed")
    output = ROOT / "models" / "decoders" / STAGE2_SCHEMA; frame.to_csv(output / "cross_validation_run_summary.csv", index=False); return frame


def run_seed_sweep():
    summaries = []
    for arm in ARMS:
        for seed in SWEEP_SEEDS:
            summaries.append(run_fold_arm(SWEEP_FOLD, arm, seed=seed, sweep=True))
    frame = pd.DataFrame(summaries)
    if not frame.success.all(): raise RuntimeError("seed-sweep contract failed")
    output = ROOT / "models" / "decoders" / STAGE2_SCHEMA / "seed_sweep"; output.mkdir(parents=True, exist_ok=True); frame.to_csv(output / "seed_sweep_run_summary.csv", index=False); return frame

In [ ]:
print("architecture contract:", architecture_contract())
print("metric tests:", metric_unit_tests())
print("block equivalence:", block_equivalence_test())
print("matched init:", matched_init_check())
print("guard tests:", guard_tests())
if RUN_CROSS_VALIDATION:
    display(run_cross_validation())
if RUN_SEED_SWEEP:
    display(run_seed_sweep())
if not (RUN_CROSS_VALIDATION or RUN_SEED_SWEEP):
    print("Definitions loaded. Set RUN_CROSS_VALIDATION=True only after all five independent QA verdicts are PASS and all five shared front ends exist; set RUN_SEED_SWEEP=True to add the fold-0 3-seed variance band.")